# 20 Video Ablation Compare

Purpose: compare whether raw video is needed, whether frozen video embeddings are enough, whether a small raw-video fine-tune helps, and whether pose/object structured features preserve enough information.

Inputs already expected from earlier notebooks:

- `clips/{FULL_RUN_ID}/clips_v1.parquet`
- `predictions/context_catboost_mlb_2024_2026_v2/metrics_v1.json`
- `predictions/video_frozen_encoder_mlb_2024_2026_v2/metrics_v1.json` if notebook 19 was run
- `predictions/sequence_tcn_mlb_2024_2026_v2/metrics_v1.json` if notebooks 16-18 were run

This notebook can also run two additional comparison baselines:

- lightweight raw video: RGB/motion statistics from the clip
- raw-video fine-tune: contact-aligned frames -> pretrained R3D-18 or tiny 3D CNN -> stat heads

Outputs:

- `predictions/video_lightweight_cv2_mlb_2024_2026_v1/*`
- `predictions/video_raw_finetune_mlb_2024_2026_v2/*`
- `reports/ablation_compare/video_ablation_mlb_2024_2026_v2/index.html`
- `reports/ablation_compare/video_ablation_mlb_2024_2026_v2/summary.json`


In [ ]:
from pathlib import Path
import os
import sys
import importlib.util


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    drive.mount(str(mountpoint))


def _is_repo_dir(path: Path) -> bool:
    return (path / 'src' / 'sport_pipeline' / '__init__.py').exists() and (path / 'configs').exists() and (path / 'notebooks').exists()


def _resolve_repo_dir() -> Path:
    fixed = Path('/content/drive/MyDrive/codex/batting_codex_handoff')
    candidates = []
    env_root = os.environ.get('BATTING_CODE_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([fixed, Path.cwd()])
    candidates.extend(Path.cwd().parents)
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            return candidate
    raise FileNotFoundError('sport_pipeline repo が見つかりません。確認した候補:\n- ' + '\n- '.join(checked))


_mount_drive_if_needed()
REPO_DIR = _resolve_repo_dir()
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME
BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)
src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(f'sport_pipeline を import できません: {src_dir}')

from sport_pipeline.pipeline.run_profile import artifact_id, load_run_profile, resolve_statcast_date_range, run_id, stage_settings

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
FULL_RUN_ID = run_id(RUN_PROFILE, 'full_run_id')
CONTEXT_RUN_ID = run_id(RUN_PROFILE, 'context_run_id')
SEQUENCE_TCN_RUN_ID = run_id(RUN_PROFILE, 'sequence_tcn_run_id')
VIDEO_LIGHTWEIGHT_RUN_ID = run_id(RUN_PROFILE, 'video_lightweight_run_id', 'video_lightweight_cv2_mlb_2024_2026_v2')
VIDEO_FROZEN_RUN_ID = run_id(RUN_PROFILE, 'video_frozen_run_id', run_id(RUN_PROFILE, 'video_run_id'))
VIDEO_FINETUNE_RUN_ID = run_id(RUN_PROFILE, 'video_finetune_run_id', 'video_raw_finetune_mlb_2024_2026_v2')
FUSION_RUN_ID = run_id(RUN_PROFILE, 'fusion_run_id')
REPORT_ID = run_id(RUN_PROFILE, 'video_ablation_report_id', 'video_ablation_mlb_2024_2026_v2')
STRUCTURED_SEQUENCE_FEATURE_ID = artifact_id(RUN_PROFILE, 'structured_sequence_feature_id')
VIDEO_LIGHTWEIGHT_FEATURE_ID = artifact_id(RUN_PROFILE, 'video_lightweight_feature_id', 'video_lightweight_features_mlb_2024_2026_v2')

print('REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('FULL_RUN_ID =', FULL_RUN_ID)
print('VIDEO_LIGHTWEIGHT_RUN_ID =', VIDEO_LIGHTWEIGHT_RUN_ID)
print('VIDEO_LIGHTWEIGHT_FEATURE_ID =', VIDEO_LIGHTWEIGHT_FEATURE_ID)
print('VIDEO_FROZEN_RUN_ID =', VIDEO_FROZEN_RUN_ID)
print('VIDEO_FINETUNE_RUN_ID =', VIDEO_FINETUNE_RUN_ID)
print('REPORT_ID =', REPORT_ID)


## Artifact Map

This cell tells you what already exists and where to inspect it.

In [ ]:
from sport_pipeline.io import read_table

RUNS = {
    'context': CONTEXT_RUN_ID,
    'raw_video_lightweight': VIDEO_LIGHTWEIGHT_RUN_ID,
    'raw_video_frozen': VIDEO_FROZEN_RUN_ID,
    'raw_video_finetune': VIDEO_FINETUNE_RUN_ID,
    'pose_object_tcn': SEQUENCE_TCN_RUN_ID,
    'fusion': FUSION_RUN_ID,
}

print('Key input/output locations')
print('clips:', BASE_DIR / 'clips' / FULL_RUN_ID / 'clips_v1.parquet')
print('clip videos dir:', BASE_DIR / 'clips' / FULL_RUN_ID / 'videos')
print('debug overlays/index:', BASE_DIR / 'debug' / FULL_RUN_ID)
print('deep CV detections:', BASE_DIR / 'detections' / FULL_RUN_ID / 'detections_v1.parquet')
print('pose2d:', BASE_DIR / 'pose2d' / FULL_RUN_ID / 'pose2d_v1.parquet')
print('structured sequence manifest:', BASE_DIR / 'features' / STRUCTURED_SEQUENCE_FEATURE_ID / 'manifest.parquet')
print('structured sequence frames:', BASE_DIR / 'features' / STRUCTURED_SEQUENCE_FEATURE_ID / 'frames.parquet')
print()

for label, rid in RUNS.items():
    pred = BASE_DIR / 'predictions' / rid / 'predictions_v1.parquet'
    metrics = BASE_DIR / 'predictions' / rid / 'metrics_v1.json'
    print(label, rid)
    print('  predictions:', pred, 'exists=', pred.exists())
    print('  metrics:', metrics, 'exists=', metrics.exists())

clips_path = BASE_DIR / 'clips' / FULL_RUN_ID / 'clips_v1.parquet'
if clips_path.exists():
    clips = read_table(clips_path)
    print('clips_v1 rows =', len(clips))
    print('clean clips =', sum(1 for row in clips if row.get('clip_status') == 'clean_clip'))
else:
    raise FileNotFoundError(f'clips_v1 がありません。12 を先に成功させてください: {clips_path}')


## Optional: Lightweight Raw Video Baseline

This is quick and useful because it separates `raw RGB clip has any signal` from `large pretrained video model helps`.

In [ ]:
from sport_pipeline.models.video.full_baseline import run_full_video_baseline

RUN_LIGHTWEIGHT_IF_MISSING = True
lightweight_metrics = BASE_DIR / 'predictions' / VIDEO_LIGHTWEIGHT_RUN_ID / 'metrics_v1.json'

if RUN_LIGHTWEIGHT_IF_MISSING and not lightweight_metrics.exists():
    VIDEO_SETTINGS = stage_settings(RUN_PROFILE, 'lightweight_video')
    outputs = run_full_video_baseline(
        BASE_DIR,
        clip_run_id=FULL_RUN_ID,
        prediction_run_id=VIDEO_LIGHTWEIGHT_RUN_ID,
        max_clips=VIDEO_SETTINGS.get('max_clips'),
        max_frames=int(VIDEO_SETTINGS.get('max_frames', 16)),
        encoder_mode='lightweight',
        require_non_empty=True,
        video_embedding_feature_id=VIDEO_LIGHTWEIGHT_FEATURE_ID,
    )
    for name, path in outputs.items():
        print(name, '->', path, 'exists=', path.exists())
else:
    print('lightweight raw video baseline は既にあります、または RUN_LIGHTWEIGHT_IF_MISSING=False です:', lightweight_metrics)


## Optional: Tiny Raw Video Fine-Tune

This is the direct `video frames -> stat heads` baseline. The real profile uses pretrained torchvision R3D-18 with a small number of epochs. Use it as an ablation, not as the final best model. L4 is fine for 500 clips; A100 is better if you raise frames/resolution/epochs.

In [ ]:
from sport_pipeline.models.video.raw_finetune import run_raw_video_finetune

FT_SETTINGS = stage_settings(RUN_PROFILE, 'raw_video_finetune')
RUN_RAW_VIDEO_FINETUNE_IF_MISSING = os.environ.get('RUN_RAW_VIDEO_FINETUNE_IF_MISSING', '1').lower() in {'1', 'true', 'yes'}
raw_video_metrics = BASE_DIR / 'predictions' / VIDEO_FINETUNE_RUN_ID / 'metrics_v1.json'
RUN_RAW_VIDEO_FINETUNE = bool(FT_SETTINGS.get('execute_default', True)) and RUN_RAW_VIDEO_FINETUNE_IF_MISSING and not raw_video_metrics.exists()

if RUN_RAW_VIDEO_FINETUNE:
    outputs = run_raw_video_finetune(
        BASE_DIR,
        clip_run_id=FULL_RUN_ID,
        prediction_run_id=VIDEO_FINETUNE_RUN_ID,
        max_clips=FT_SETTINGS.get('max_clips', 500),
        num_frames=int(FT_SETTINGS.get('num_frames', 16)),
        image_size=int(FT_SETTINGS.get('image_size', 112)),
        batch_size=int(FT_SETTINGS.get('batch_size', 4)),
        max_epochs=int(FT_SETTINGS.get('max_epochs', 5)),
        learning_rate=float(FT_SETTINGS.get('learning_rate', 1e-4)),
        model_family=str(FT_SETTINGS.get('model_family', 'tiny3d')),
        pretrained=bool(FT_SETTINGS.get('pretrained', False)),
        freeze_backbone=bool(FT_SETTINGS.get('freeze_backbone', False)),
        allow_model_download=bool(FT_SETTINGS.get('allow_model_download', False)),
        hidden_dim=int(FT_SETTINGS.get('hidden_dim', 128)),
        dropout=float(FT_SETTINGS.get('dropout', 0.2)),
        device=str(FT_SETTINGS.get('device', 'auto')),
        require_non_empty=bool(FT_SETTINGS.get('require_non_empty', True)),
        resume=bool(FT_SETTINGS.get('resume', True)),
    )
    for name, path in outputs.items():
        print(name, '->', path, 'exists=', path.exists())
else:
    print('raw-video fine-tune は skip しました。')
    print('raw_video_metrics =', raw_video_metrics, 'exists=', raw_video_metrics.exists())
    print('RUN_RAW_VIDEO_FINETUNE_IF_MISSING =', RUN_RAW_VIDEO_FINETUNE_IF_MISSING)
    print('設定場所:', RUN_PROFILE_PATH, 'execution.raw_video_finetune.execute_default')


## Build Ablation Report

In [ ]:
from IPython.display import IFrame, display
from sport_pipeline.reports.ablation_compare import build_video_ablation_report, run_specs_from_profile

outputs = build_video_ablation_report(BASE_DIR, report_id=REPORT_ID, run_specs=run_specs_from_profile(RUN_PROFILE))
for name, path in outputs.items():
    print(name, '->', path, 'exists=', path.exists())

report_path = outputs['html']
print('Open this report from Drive:', report_path)
if report_path.exists():
    display(IFrame(str(report_path), width='100%', height=900))
